
# BigQuery Fundamentals — Indian Employee Dataset

IndexCorp wants its HR data in BigQuery — headcount by department, salary by gender/state,
attendance trends — and wants it to stay fast and cheap as attendance history grows into the
millions of rows. Same journey as the bikeshare version of this lab, different data: loading,
queries, joins, views, materialized views, partitioning, clustering, one concept at a time.

## The data
Three CSVs on GitHub — `departments` (8 rows), `employees` (100 rows), `attendance_monthly`
(1,200 rows: 100 employees × 12 months, the only one with a real time dimension, which is why
partitioning/clustering live there).

Source: `github.com/himanshurathi/gcp-training-labs-accordion/tree/master/day-1-foundations`

| Table | Key columns |
|---|---|
| `departments` | `department_id`, `department_name`, `location`, `department_head` |
| `employees` | `employee_id`, name/gender/state/city, `department_id`, `designation`, `date_of_joining`, `monthly_salary_inr` |
| `attendance_monthly` | `employee_id`, `department_id`/`state` (denormalized on purpose — saves a join in every partition/cluster query later), `month_date`, `days_present`, `leaves_taken`, `performance_rating` |

Fully self-contained: auth → configure → CSVs pull straight from GitHub, no upload step.


## 1. Auth

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("authenticated")
else:
    print("not in Colab — assuming ADC is set up")


## 2. Config + a `hr_analytics` dataset to hold everything

In [ ]:
PROJECT_ID = input("Enter your GCP Project ID: ").strip()
LOCATION = input("BigQuery location [US]: ").strip() or "US"
DATASET_ID = "hr_analytics"

!gcloud config set project {PROJECT_ID}
!gcloud services enable bigquery.googleapis.com --project {PROJECT_ID}

from google.cloud import bigquery
from google.cloud.exceptions import Conflict
import pandas as pd

bq_client = bigquery.Client(project=PROJECT_ID)

def TBL(name):
    return f"`{PROJECT_ID}.{DATASET_ID}.{name}`"

def run_query(sql, label=None, expect_rows=True):
    if label:
        print(f"--- {label} ---")
    job = bq_client.query(sql)
    if expect_rows:
        try:
            display(job.result().to_dataframe())
        except Exception:
            job.result()
    else:
        job.result()
    return job

try:
    bq_client.create_dataset(bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}"), exists_ok=False)
    print("created hr_analytics")
except Conflict:
    print("hr_analytics already exists — reusing it")



> 🖥️ SQL workspace should show the new `hr_analytics` dataset, empty for now.



## 3. Pull the CSVs straight from GitHub
`pandas.read_csv` on the raw URL, then `load_table_from_dataframe` — schema gets inferred from the
DataFrame's dtypes. Fine for clean files like these; for anything production-bound, declare an
explicit schema instead so one malformed row can't quietly change a column's type.


In [ ]:
GITHUB_BASE = "https://raw.githubusercontent.com/himanshurathi/gcp-training-labs-accordion/master/day-1-foundations"

departments_df = pd.read_csv(f"{GITHUB_BASE}/departments.csv")
# parse_dates alone isn't quite enough: it gives pandas a datetime64 (a full timestamp), which
# BigQuery then loads as TIMESTAMP/DATETIME, not DATE -- and bare `PARTITION BY month_date` later
# on only works against a true DATE column. Converting to .dt.date gives real date objects, which
# load into BigQuery as DATE.
employees_df = pd.read_csv(f"{GITHUB_BASE}/employees.csv", parse_dates=["date_of_joining"])
employees_df["date_of_joining"] = employees_df["date_of_joining"].dt.date

attendance_df = pd.read_csv(f"{GITHUB_BASE}/attendance_monthly.csv", parse_dates=["month_date"])
attendance_df["month_date"] = attendance_df["month_date"].dt.date

load_cfg = bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)

for name, df in [("departments", departments_df), ("employees", employees_df), ("attendance_monthly", attendance_df)]:
    bq_client.load_table_from_dataframe(df, TBL(name).strip("`"), job_config=load_cfg).result()
    print(name, "loaded:", bq_client.get_table(TBL(name).strip("`")).num_rows, "rows")

# confirm the types actually landed right -- catch this here, not three sections later
schema = bq_client.get_table(TBL("attendance_monthly").strip("`")).schema
month_date_type = next(f.field_type for f in schema if f.name == "month_date")
print(f"\nmonth_date loaded as: {month_date_type}")
assert month_date_type == "DATE", (
    f"month_date loaded as {month_date_type}, not DATE -- bare PARTITION BY month_date in "
    f"Section 8 requires an actual DATE column, not TIMESTAMP/DATETIME"
)



> 🖥️ `hr_analytics` should now list all three tables — click one and check the Schema tab against
> what got inferred, and Preview for the actual rows.



## 4. Basic queries
"Karnataka employees earning over ₹80,000, highest first" — filter, sort, done.


In [ ]:
run_query(f'''
SELECT employee_id, first_name, last_name, state, designation, monthly_salary_inr
FROM {TBL("employees")}
WHERE state = 'Karnataka' AND monthly_salary_inr > 80000
ORDER BY monthly_salary_inr DESC
''', label="Karnataka, >80k")


In [ ]:
run_query(f'''
SELECT COUNT(*) AS total_employees, ROUND(AVG(monthly_salary_inr), 2) AS avg_salary,
       MIN(monthly_salary_inr) AS min_salary, MAX(monthly_salary_inr) AS max_salary
FROM {TBL("employees")}
''', label="company-wide salary stats")

run_query(f'''
SELECT state, COUNT(*) AS headcount, ROUND(AVG(monthly_salary_inr), 2) AS avg_salary
FROM {TBL("employees")}
GROUP BY state
ORDER BY headcount DESC
''', label="headcount + avg salary by state")



> 🖥️ Run either in the SQL workspace, then click a column header in the results grid to re-sort
> without re-running the query.


In [ ]:
# window function: same "average per department" idea as GROUP BY, but keeps one row per employee
# instead of collapsing to one row per department
run_query(f'''
SELECT
  employee_id, first_name, department_id, monthly_salary_inr,
  ROUND(AVG(monthly_salary_inr) OVER (PARTITION BY department_id), 2) AS dept_avg_salary,
  ROUND(monthly_salary_inr - AVG(monthly_salary_inr) OVER (PARTITION BY department_id), 2) AS diff_from_dept_avg
FROM {TBL("employees")}
ORDER BY department_id, diff_from_dept_avg DESC
''', label="salary vs department average")



## 5. Joins
Employees only have a `department_id` number — the readable name lives in `departments`.


In [ ]:
run_query(f'''
SELECT e.employee_id, e.first_name, e.last_name, d.department_name, d.location
FROM {TBL("employees")} AS e
INNER JOIN {TBL("departments")} AS d ON e.department_id = d.department_id
LIMIT 20
''', label="employees + department details")


In [ ]:
run_query(f'''
SELECT d.department_name, d.location, COUNT(e.employee_id) AS headcount,
       ROUND(AVG(e.monthly_salary_inr), 2) AS avg_salary
FROM {TBL("departments")} AS d
INNER JOIN {TBL("employees")} AS e ON d.department_id = e.department_id
GROUP BY d.department_name, d.location
ORDER BY avg_salary DESC
''', label="avg salary per department")



Any employee whose `department_id` doesn't match anything gets silently dropped by an INNER JOIN —
LEFT JOIN + `WHERE right_side IS NULL` finds departments with *zero* staff, the mirror case:


In [ ]:
run_query(f'''
SELECT d.department_id, d.department_name
FROM {TBL("departments")} AS d
LEFT JOIN {TBL("employees")} AS e ON d.department_id = e.department_id
WHERE e.employee_id IS NULL
''', label="departments with no employees")



> 🖥️ Flip the LEFT JOIN back to INNER and re-run — any empty-department row (if one exists) just
> disappears. That's the whole difference between the two, made concrete.



## 6. Views
Stores the query, not the data — every read re-runs the join+aggregation against live tables.


In [ ]:
run_query(f'''
CREATE OR REPLACE VIEW {TBL("department_summary_view")} AS
SELECT d.department_name, d.location, COUNT(e.employee_id) AS headcount,
       ROUND(AVG(e.monthly_salary_inr), 2) AS avg_salary
FROM {TBL("departments")} AS d
INNER JOIN {TBL("employees")} AS e ON d.department_id = e.department_id
GROUP BY d.department_name, d.location
''', label="create view", expect_rows=False)

run_query(f'SELECT * FROM {TBL("department_summary_view")} ORDER BY avg_salary DESC', label="query it like a table")



> 🖥️ `department_summary_view` shows a "view" icon in the Explorer; its Details tab shows
> Table type: VIEW and the stored SQL — no rows of its own.

Always live, zero storage cost, zero performance win — every read redoes the whole join. That's
what a materialized view is for.


## 7. Materialized views

In [ ]:
run_query(f'''
CREATE MATERIALIZED VIEW {TBL("monthly_attendance_summary_mv")} AS
SELECT a.month_date, a.department_id,
       AVG(a.days_present) AS avg_days_present,
       AVG(a.performance_rating) AS avg_performance_rating,
       COUNT(*) AS employee_month_records
FROM {TBL("attendance_monthly")} AS a
GROUP BY a.month_date, a.department_id
''', label="create MV", expect_rows=False)

# incremental MVs must output raw aggregates (no ROUND() inside the MV itself) --
# round here instead, now that we're reading the stored values back
run_query(f'''
SELECT month_date, department_id,
       ROUND(avg_days_present, 2) AS avg_days_present,
       ROUND(avg_performance_rating, 2) AS avg_performance_rating,
       employee_month_records
FROM {TBL("monthly_attendance_summary_mv")}
WHERE month_date = '2025-06-01'
ORDER BY avg_performance_rating DESC
''', label="query it")



> 🖥️ Distinct "materialized view" icon; Details tab has a Refresh section a plain view never
> shows, since it has nothing to refresh.

Restricted SQL subset (no `CURRENT_TIMESTAMP()`, limited joins — check current docs), near-real-
time not instant, and unlike a plain view it does cost storage.



## 8. Partitioning
`attendance_monthly` is 1,200 rows here — trivial either way. Imagine the same table at 5 years and
10,000 employees: millions of rows, and a "just June" query shouldn't have to touch the rest.


In [ ]:
run_query(f'''
CREATE TABLE {TBL("attendance_monthly_partitioned")}
PARTITION BY month_date AS
SELECT * FROM {TBL("attendance_monthly")}
''', label="create partitioned copy", expect_rows=False)


In [ ]:
unpart = bq_client.query(
    f'SELECT AVG(days_present) FROM {TBL("attendance_monthly")} WHERE month_date = "2025-06-01"',
    job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False),
)
part = bq_client.query(
    f'SELECT AVG(days_present) FROM {TBL("attendance_monthly_partitioned")} WHERE month_date = "2025-06-01"',
    job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False),
)
print(f"unpartitioned: {unpart.total_bytes_processed:,} bytes")
print(f"partitioned:   {part.total_bytes_processed:,} bytes")



> 🖥️ `attendance_monthly_partitioned`'s Details tab confirms partitioning by `month_date`, with
> per-partition row counts under Table storage.

At this size the gap looks unremarkable — the same pruning is what turns a 500GB scan into a 2GB
one once the table is actually large. The mechanism is the point, not today's numbers.



## 9. Clustering
Partitioning gets you to a month. Clustering sorts *within* each partition so a department filter
can skip blocks too.


In [ ]:
run_query(f'''
CREATE TABLE {TBL("attendance_monthly_partitioned_clustered")}
PARTITION BY month_date
CLUSTER BY department_id AS
SELECT * FROM {TBL("attendance_monthly")}
''', label="create partitioned + clustered copy", expect_rows=False)


In [ ]:
run_query(f'''
SELECT AVG(performance_rating) FROM {TBL("attendance_monthly_partitioned")}
WHERE month_date = '2025-06-01' AND department_id = 101
''', label="partitioned only")

run_query(f'''
SELECT AVG(performance_rating) FROM {TBL("attendance_monthly_partitioned_clustered")}
WHERE month_date = '2025-06-01' AND department_id = 101
''', label="partitioned + clustered")



> 🖥️ The clustered table's Details tab shows both a Partition section and a Clustering section —
> the one place you see both together.

Clustering's payoff doesn't always show cleanly in a dry-run estimate the way partition pruning
does. For a real answer, run both for real and compare Bytes billed, not the estimate.


## Cleanup

In [ ]:
CONFIRM_DELETE = False  # flip to True when ready

if not CONFIRM_DELETE:
    print("CONFIRM_DELETE is False — nothing happens. Flip it and re-run when ready.")


In [ ]:
if CONFIRM_DELETE:
    bq_client.delete_dataset(f"{PROJECT_ID}.{DATASET_ID}", delete_contents=True, not_found_ok=True)
    print(f"{DATASET_ID} and everything in it: gone")
